In [1]:

from pathlib import Path
import re
import difflib
import subprocess

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, Markdown, clear_output

# Source of truth: existing firewall configuration.
SOURCE_FILE = Path("/etc/nft/web01/rules/10-tcp-networks.nft")

# Save button writes here, never directly to the production file.
STAGING_FILE = Path("/tmp/10-tcp-networks.zone01.staged.nft")

R01 = "Z01_TCP4_R01"
R02 = "Z01_TCP4_R02"

DEFAULTS = {
    "interface": "$nic_lan",
    "source": "$ip_lan",
    "destination": "$net_lan01",
    "port": "",
    "verdict": "accept",
}


def read_source():
    if not SOURCE_FILE.is_file():
        raise FileNotFoundError(
            f"Cannot read source file: {SOURCE_FILE}"
        )
    return SOURCE_FILE.read_text()


def parse_definition(source, name):
    pattern = (
        r"(?m)^\s*define\s+" + re.escape(name)
        + r"\s*=\s*\{(.*?)^\s*\}"
    )
    match = re.search(pattern, source, flags=re.DOTALL)

    if not match:
        raise ValueError(f"Definition {name} was not found.")

    body = match.group(1)
    entries = []

    for line_number, raw_line in enumerate(body.splitlines(), start=1):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        # Strip an optional trailing comma.
        line = line.rstrip(",").strip()

        if ":" not in line:
            raise ValueError(
                f"Cannot parse {name}, body line {line_number}: {raw_line}"
            )

        key_text, verdict = line.rsplit(":", 1)
        fields = [part.strip() for part in key_text.split(".")]
        verdict = verdict.strip()

        entries.append({
            "fields": fields,
            "verdict": verdict,
        })

    return entries


def load_rules():
    source = read_source()

    r01_entries = parse_definition(source, R01)
    r02_entries = parse_definition(source, R02)

    if len(r01_entries) != len(r02_entries):
        raise ValueError(
            f"{R01} has {len(r01_entries)} rules but "
            f"{R02} has {len(r02_entries)}. "
            "The editor requires one-to-one correspondence."
        )

    rules = []

    for index, (a, b) in enumerate(
        zip(r01_entries, r02_entries), start=1
    ):
        fields01 = a["fields"]
        fields02 = b["fields"]

        if len(fields01) != 5 or len(fields02) != 4:
            raise ValueError(
                f"Rule {index}: expected 5 R01 key fields and "
                "4 R02 key fields."
            )

        if fields01[0] != "$Z01":
            raise ValueError(
                f"Rule {index}: unexpected R01 zone key "
                f"{fields01[0]!r}."
            )

        if fields01[1:] != fields02:
            raise ValueError(
                f"Rule {index}: R01/R02 keys do not correspond.\n"
                f"R01: {fields01}\nR02: {fields02}"
            )

        if a["verdict"] != b["verdict"]:
            raise ValueError(
                f"Rule {index}: R01/R02 verdicts differ."
            )

        rules.append({
            "interface": fields01[1],
            "source": fields01[2],
            "destination": fields01[3],
            "port": fields01[4],
            "verdict": a["verdict"],
        })

    return source, rules


source_original, rules = load_rules()

print(f"Source: {SOURCE_FILE}")
print(f"Staging destination: {STAGING_FILE}")
print(f"Zone 01 TCP rules loaded: {len(rules)}")


Source: /etc/nft/web01/rules/10-tcp-networks.nft
Staging destination: /tmp/10-tcp-networks.zone01.staged.nft
Zone 01 TCP rules loaded: 10


In [2]:

# ============================================================
# Rule rendering
# ============================================================

def key_r01(rule):
    fields = [
        "$Z01",
        rule["interface"],
        rule["source"],
        rule["destination"],
        rule["port"],
    ]
    return " . ".join(fields)


def key_r02(rule):
    fields = [
        rule["interface"],
        rule["source"],
        rule["destination"],
        rule["port"],
    ]
    return " . ".join(fields)


def nft_entry(key, verdict):
    return f"{key} : {verdict},"


def rule_table():
    rows = []

    for number, rule in enumerate(rules, start=1):
        rows.append({
            "#": number,
            "Interface": rule["interface"],
            "Source": rule["source"],
            "Destination": rule["destination"],
            "Port": rule["port"],
            "Verdict": rule["verdict"],
            "R01 entry": nft_entry(key_r01(rule), rule["verdict"]),
            "R02 entry": nft_entry(key_r02(rule), rule["verdict"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# Widgets
# ============================================================

rule_selector = widgets.Dropdown(
    options=[],
    description="Rule:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "initial"},
)

interface_input = widgets.Text(
    value=DEFAULTS["interface"],
    description="Interface:",
    layout=widgets.Layout(width="280px"),
)

source_input = widgets.Text(
    value=DEFAULTS["source"],
    description="Source:",
    layout=widgets.Layout(width="280px"),
)

destination_input = widgets.Text(
    value=DEFAULTS["destination"],
    description="Destination:",
    layout=widgets.Layout(width="280px"),
)

port_input = widgets.Text(
    value="",
    placeholder="e.g. 443 or $https",
    description="Destination port:",
    layout=widgets.Layout(width="280px"),
)

verdict_input = widgets.Dropdown(
    options=["accept", "drop", "reject"],
    value="accept",
    description="Verdict:",
    layout=widgets.Layout(width="280px"),
)

add_button = widgets.Button(
    description="Add",
    icon="plus",
    button_style="success",
)

edit_button = widgets.Button(
    description="Update",
    icon="check",
    button_style="primary",
)

delete_button = widgets.Button(
    description="Delete",
    icon="trash",
    button_style="danger",
)

save_button = widgets.Button(
    description="Save staging file",
    icon="save",
    button_style="info",
)

table_output = widgets.Output()
status_output = widgets.Output()
preview_output = widgets.Output()


def set_status(message, error=False):
    with status_output:
        clear_output(wait=True)
        display(HTML(
            f"<p style='color:{'#b91c1c' if error else '#166534'}'>"
            f"{message}</p>"
        ))


def refresh_table():
    options = [
        (f"Rule {i}: {rule['port']}", i - 1)
        for i, rule in enumerate(rules, start=1)
    ]

    previous = rule_selector.value
    rule_selector.options = options

    if options:
        valid_values = [value for _, value in options]
        rule_selector.value = (
            previous if previous in valid_values
            else valid_values[0]
        )

    with table_output:
        clear_output(wait=True)
        display(Markdown("### Current TCP Zone 01 rules"))
        display(
            rule_table().style
            .set_properties(**{
                "text-align": "left",
                "white-space": "pre-wrap",
                "overflow-wrap": "anywhere",
                "min-width": "180px",
            })
            .set_table_styles([
                {"selector": "table", "props": [
                    ("width", "100%"),
                    ("table-layout", "auto"),
                ]},
                {"selector": "td", "props": [
                    ("max-width", "650px"),
                    ("vertical-align", "top"),
                ]},
            ])
        )


def load_selected_rule(*_):
    index = rule_selector.value

    if index is None or index >= len(rules):
        return

    rule = rules[index]
    interface_input.value = rule["interface"]
    source_input.value = rule["source"]
    destination_input.value = rule["destination"]
    port_input.value = rule["port"]
    verdict_input.value = rule["verdict"]


def form_rule():
    rule = {
        "interface": interface_input.value.strip(),
        "source": source_input.value.strip(),
        "destination": destination_input.value.strip(),
        "port": port_input.value.strip(),
        "verdict": verdict_input.value,
    }

    for field, value in rule.items():
        if not value:
            raise ValueError(f"{field} cannot be empty.")

        if any(c in value for c in ("\n", "\r", ",", "{", "}")):
            raise ValueError(
                f"Invalid character in {field}: {value!r}"
            )

    # These fields are nftables expressions, not arbitrary free text.
    # Reject separators that could create additional rule expressions.
    for field in ("interface", "source", "destination", "port"):
        value = rule[field]
        if ":" in value or ";" in value:
            raise ValueError(
                f"Unexpected ':' or ';' in {field}."
            )

    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", rule["verdict"]):
        raise ValueError("Invalid verdict.")

    return rule


def add_rule(_):
    try:
        rule = form_rule()

        # Prevent duplicate logical entries.
        if rule in rules:
            raise ValueError("This rule already exists.")

        rules.append(rule)
        refresh_table()
        load_selected_rule()
        set_status("Rule added to the in-memory staging model.")
    except Exception as exc:
        set_status(str(exc), error=True)


def update_rule(_):
    try:
        index = rule_selector.value
        if index is None:
            raise ValueError("Select a rule to update.")

        updated = form_rule()

        if any(
            i != index and other == updated
            for i, other in enumerate(rules)
        ):
            raise ValueError("Another rule already has these fields.")

        rules[index] = updated
        refresh_table()
        set_status("Rule updated in the in-memory staging model.")
    except Exception as exc:
        set_status(str(exc), error=True)


def delete_rule(_):
    index = rule_selector.value

    if index is None:
        set_status("Select a rule to delete.", error=True)
        return

    removed = rules.pop(index)
    refresh_table()
    set_status(f"Deleted logical rule for port {removed['port']}.")


rule_selector.observe(load_selected_rule, names="value")
add_button.on_click(add_rule)
edit_button.on_click(update_rule)
delete_button.on_click(delete_rule)

display(Markdown("# TCP Zone 01 — Rule Editor"))
display(Markdown(
    "One row represents both R01 and R02. Changes remain in memory "
    "until saved to the staging file."
))

display(widgets.VBox([
    table_output,
    widgets.HTML("<h3>Rule editor</h3>"),
    rule_selector,
    widgets.HBox([interface_input, source_input]),
    widgets.HBox([destination_input, port_input]),
    verdict_input,
    widgets.HBox([add_button, edit_button, delete_button]),
    status_output,
]))

refresh_table()
load_selected_rule()


# TCP Zone 01 — Rule Editor

One row represents both R01 and R02. Changes remain in memory until saved to the staging file.

In [5]:

# TCP Zone 01 — Validate, stage, test, deploy, and reload

import difflib
import re
import shutil
import subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SOURCE_FILE = Path("/etc/nft/web01/rules/10-tcp-networks.nft")
STAGING_FILE = Path("/tmp/10-tcp-network.nft")
RULESET_ENTRYPOINT = Path("/tmp/nftables-zone01-test.conf")
PRODUCTION_ENTRYPOINT = Path("/etc/nftables.conf")
VALIDATION_NAMESPACE = "web01"

VARIABLES_INCLUDE = "/etc/nft/web01/variables/*.nft"
CONSTANTS_INCLUDE = "/etc/nft/web01/constants/*.nft"
ICMP_INCLUDE = "/etc/nft/web01/rules/06-icmp-networks.nft"
UDP_INCLUDE = "/etc/nft/web01/rules/08-udp-networks.nft"
ZONES_INCLUDE = "/etc/nft/web01/zones/*.nft"


# ------------------------------------------------------------
# Logical validation
# ------------------------------------------------------------

def validate_zone01_rules():
    errors = []
    warnings = []
    seen = set()

    required = ("interface", "source", "destination", "port", "verdict")

    if not rules:
        warnings.append("Both Zone 01 TCP definitions will be empty.")

    for index, rule in enumerate(rules, start=1):
        label = f"Rule {index}"

        for field in required:
            value = rule.get(field, "")
            if not isinstance(value, str) or not value.strip():
                errors.append(f"{label}: {field} is empty.")

        port = str(rule.get("port", "")).strip()
        if port:
            if port.isdigit():
                if not 1 <= int(port) <= 65535:
                    errors.append(
                        f"{label}: port must be between 1 and 65535."
                    )
            elif not re.fullmatch(r"\$[A-Za-z_][A-Za-z0-9_]*", port):
                errors.append(
                    f"{label}: port must be numeric or a variable "
                    "such as $https."
                )

        for field in required:
            value = str(rule.get(field, ""))
            if any(ch in value for ch in ("\n", "\r", ";", "{", "}")):
                errors.append(f"{label}: invalid delimiter in {field}.")

        verdict = rule.get("verdict", "")
        if verdict not in {"accept", "drop", "reject"}:
            errors.append(f"{label}: unsupported verdict {verdict!r}.")

        signature = tuple(
            str(rule.get(field, "")).strip() for field in required
        )
        if signature in seen:
            errors.append(f"{label}: duplicate logical rule.")
        seen.add(signature)

        expected_r01 = ["$Z01"] + key_r02(rule).split(" . ")
        actual_r01 = key_r01(rule).split(" . ")

        if actual_r01 != expected_r01:
            errors.append(f"{label}: R01/R02 keys do not correspond.")

    return errors, warnings


# ------------------------------------------------------------
# Generate both map definitions in the staged TCP file
# ------------------------------------------------------------

def render_zone01_definition(name, entries):
    lines = [f"define {name} = {{"]

    for entry in entries:
        lines.append(
            f"    {nft_entry(entry['key'], entry['verdict'])}"
        )

    lines.append("}")
    return "\n".join(lines)


def build_zone01_staged_source():
    current_source = read_source()

    if current_source != source_original:
        raise RuntimeError(
            "Production source changed since it was loaded. "
            "Reload and reconcile changes before proceeding."
        )

    replacements = {
        R01: render_zone01_definition(
            R01,
            [
                {"key": key_r01(rule), "verdict": rule["verdict"]}
                for rule in rules
            ],
        ),
        R02: render_zone01_definition(
            R02,
            [
                {"key": key_r02(rule), "verdict": rule["verdict"]}
                for rule in rules
            ],
        ),
    }

    staged = current_source

    for name, replacement in replacements.items():
        pattern = (
            r"(?m)^[ \t]*define[ \t]+" + re.escape(name)
            + r"[ \t]*=[ \t]*\{.*?^[ \t]*\}"
        )

        staged, count = re.subn(
            pattern,
            lambda match, replacement=replacement: replacement,
            staged,
            count=1,
            flags=re.DOTALL,
        )

        if count != 1:
            raise RuntimeError(
                f"Expected exactly one definition for {name}; found {count}."
            )

    return staged


# ------------------------------------------------------------
# Build isolated test entry point
# ------------------------------------------------------------

def build_test_entrypoint():
    required_files = [
        Path(ICMP_INCLUDE),
        Path(UDP_INCLUDE),
    ]

    missing = [
        str(path) for path in required_files if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Required rule files are missing: " + ", ".join(missing)
        )

    if not STAGING_FILE.is_file():
        raise FileNotFoundError(f"Staged TCP file missing: {STAGING_FILE}")

    text = "\n".join([
        "# Generated by the TCP Zone 01 editor.",
        "# Syntax-check only; do not load this test file directly.",
        "flush ruleset",
        f'include "{VARIABLES_INCLUDE}"',
        f'include "{CONSTANTS_INCLUDE}"',
        f'include "{ICMP_INCLUDE}"',
        f'include "{UDP_INCLUDE}"',
        f'include "{STAGING_FILE}"',
        f'include "{ZONES_INCLUDE}"',
        "",
    ])

    RULESET_ENTRYPOINT.write_text(text)
    return RULESET_ENTRYPOINT


# ------------------------------------------------------------
# Test the candidate ruleset without applying it
# ------------------------------------------------------------

def check_staged_configuration():
    if shutil.which("ip") is None:
        return {
            "status": "failed",
            "message": "The ip command was not found.",
        }

    if shutil.which("nft") is None:
        return {
            "status": "failed",
            "message": "The nft command was not found.",
        }

    try:
        entrypoint = build_test_entrypoint()

        command = [
            "ip", "netns", "exec", VALIDATION_NAMESPACE,
            "nft", "-c", "-f", str(entrypoint),
        ]

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )

        output = "\n".join(
            part.strip()
            for part in (result.stdout, result.stderr)
            if part.strip()
        )

        if result.returncode == 0:
            return {
                "status": "passed",
                "message": (
                    "Candidate ruleset passed nft -c. No rules applied."
                    + (f"\n{output}" if output else "")
                ),
            }

        return {
            "status": "failed",
            "message": (
                f"nft -c failed (exit {result.returncode}).\n{output}"
            ),
        }

    except subprocess.TimeoutExpired:
        return {"status": "failed", "message": "Syntax check timed out."}
    except (OSError, RuntimeError, ValueError) as exc:
        return {"status": "failed", "message": str(exc)}


# ------------------------------------------------------------
# Deploy and reload only after a successful syntax check
# ------------------------------------------------------------

def deploy_and_reload():
    if not PRODUCTION_ENTRYPOINT.is_file():
        raise FileNotFoundError(
            f"Production entry point not found: {PRODUCTION_ENTRYPOINT}"
        )

    if read_source() != source_original:
        raise RuntimeError(
            "Production TCP file changed externally; deployment cancelled."
        )

    shutil.copyfile(STAGING_FILE, SOURCE_FILE)

    command = [
        "ip", "netns", "exec", VALIDATION_NAMESPACE,
        "nft", "-f", str(PRODUCTION_ENTRYPOINT),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    output = "\n".join(
        part.strip()
        for part in (result.stdout, result.stderr)
        if part.strip()
    )

    if result.returncode != 0:
        raise RuntimeError(
            "The TCP file was replaced, but the production reload failed "
            f"(exit {result.returncode}).\n{output}\n"
            "Inspect the namespace ruleset and resolve the reload failure."
        )

    return output or "Production ruleset reloaded successfully."


# ------------------------------------------------------------
# Single action: validate -> stage -> test -> deploy -> reload
# ------------------------------------------------------------

validation_output = widgets.Output()
diff_output = widgets.Output()

workflow_button = widgets.Button(
    description="Validate, test, deploy and reload",
    icon="check",
    button_style="warning",
)


def run_zone01_workflow(_):
    global source_original

    with validation_output:
        clear_output(wait=True)

        errors, warnings = validate_zone01_rules()
        display(Markdown("## Validation report"))

        if errors:
            display(HTML(
                "<h4 style='color:#b91c1c'>Validation failed</h4>"
            ))
            for error in errors:
                print("ERROR:", error)

            print("\nNo staging or deployment performed.")
            set_status("Validation failed; deployment cancelled.", error=True)
            return

        display(HTML(
            "<h4 style='color:#166534'>Logical validation passed</h4>"
        ))

        for warning in warnings:
            print("WARNING:", warning)

        try:
            staged_text = build_zone01_staged_source()
            STAGING_FILE.parent.mkdir(parents=True, exist_ok=True)
            STAGING_FILE.write_text(staged_text)

            print(f"Staging file: {STAGING_FILE}")
            print(f"Test entry point: {RULESET_ENTRYPOINT}")
            print(f"Logical rules: {len(rules)}")
            print(f"R01 entries: {len(rules)}")
            print(f"R02 entries: {len(rules)}")

            check = check_staged_configuration()
            print(f"\nnftables syntax check: {check['status']}")
            print(check["message"])

            with diff_output:
                clear_output(wait=True)
                diff = difflib.unified_diff(
                    source_original.splitlines(),
                    staged_text.splitlines(),
                    fromfile=str(SOURCE_FILE),
                    tofile=str(STAGING_FILE),
                    lineterm="",
                )
                display(Markdown("## Proposed TCP rules diff"))
                print("\n".join(diff) or "No changes detected.")

            if check["status"] != "passed":
                set_status(
                    "Syntax check did not pass. Deployment cancelled.",
                    error=True,
                )
                print(
                    "\nProduction TCP file and ruleset were not changed."
                )
                return

            reload_output = deploy_and_reload()

            # Update the concurrency guard after successful deployment.
            source_original = read_source()

            print(f"\nDeployed TCP file: {SOURCE_FILE}")
            print(f"Reloaded entry point: {PRODUCTION_ENTRYPOINT}")
            print(f"Namespace: {VALIDATION_NAMESPACE}")
            print(reload_output)

            set_status(
                "Syntax check passed; TCP rules deployed and ruleset reloaded."
            )

        except Exception as exc:
            print(f"\nERROR: {exc}")
            set_status(f"Workflow stopped: {exc}", error=True)


workflow_button.on_click(run_zone01_workflow)

display(Markdown("# TCP Zone 01 — Complete deployment workflow"))
display(Markdown(
    "Logical validation and the isolated nftables syntax check must pass "
    "before the production TCP file is replaced and the ruleset reloaded. "
    "No backups are created."
))
display(workflow_button)
display(validation_output)
display(diff_output)


# TCP Zone 01 — Complete deployment workflow

Logical validation and the isolated nftables syntax check must pass before the production TCP file is replaced and the ruleset reloaded. No backups are created.

Button(button_style='warning', description='Validate, test, deploy and reload', icon='check', style=ButtonStyl…

Output()

Output()